# EXPERIMENT-5 : Discretization and binning

## 5.1 Discretization and Binning

In [3]:
import numpy as np
import pandas as pd

In [4]:
height = [120, 122, 125, 127, 121, 123, 137, 131, 161, 145, 141, 132] 
bins = [118, 125, 135, 160, 200] 
category = pd.cut(height, bins) 
print("Binned Intervals Result:")
print(category, "\n")

Binned Intervals Result:
[(118, 125], (118, 125], (118, 125], (125, 135], (118, 125], ..., (125, 135], (160, 200], (135, 160], (135, 160], (125, 135]]
Length: 12
Categories (4, interval[int64, right]): [(118, 125] < (125, 135] < (135, 160] < (160, 200]] 



In [5]:
bin_counts = pd.Series(category).value_counts()
print(bin_counts, "\n")

(118, 125]    5
(125, 135]    3
(135, 160]    3
(160, 200]    1
Name: count, dtype: int64 



In [6]:
bin_names = ['Short Height', 'Average height', 'Good Height', 'Taller'] 
labeled_category = pd.cut(height, bins, labels=bin_names) 
print("Labeled Bins Result:")
print(labeled_category, "\n")

Labeled Bins Result:
['Short Height', 'Short Height', 'Short Height', 'Average height', 'Short Height', ..., 'Average height', 'Taller', 'Good Height', 'Good Height', 'Average height']
Length: 12
Categories (4, object): ['Short Height' < 'Average height' < 'Good Height' < 'Taller'] 



In [7]:
np.random.seed(42) 
random_data_1 = np.random.rand(40)
auto_bins = pd.cut(random_data_1, 5, precision=2)
print("Auto 5-Bin Cut Result:")
print(auto_bins, "\n")

Auto 5-Bin Cut Result:
[(0.21, 0.4], (0.78, 0.97], (0.59, 0.78], (0.59, 0.78], (0.02, 0.21], ..., (0.78, 0.97], (0.21, 0.4], (0.02, 0.21], (0.59, 0.78], (0.4, 0.59]]
Length: 40
Categories (5, interval[float64, right]): [(0.02, 0.21] < (0.21, 0.4] < (0.4, 0.59] < (0.59, 0.78] < (0.78, 0.97]] 



In [8]:
random_data_2 = np.random.rand(2000)
category3 = pd.qcut(random_data_2, 4) 
print("Quartile Cut (First 10 values shown):")
print(category3[:10], "\n")

Quartile Cut (First 10 values shown):
[(0.00222, 0.239], (0.239, 0.506], (0.00222, 0.239], (0.751, 1.0], (0.239, 0.506], (0.506, 0.751], (0.239, 0.506], (0.506, 0.751], (0.506, 0.751], (0.00222, 0.239]]
Categories (4, interval[float64, right]): [(0.00222, 0.239] < (0.239, 0.506] < (0.506, 0.751] < (0.751, 1.0]] 



In [9]:
quantile_counts = pd.Series(category3).value_counts()
print(quantile_counts)

(0.00222, 0.239]    500
(0.239, 0.506]      500
(0.506, 0.751]      500
(0.751, 1.0]        500
Name: count, dtype: int64


## 5.2 Outlier Detection and Filtering

In [10]:
import numpy as np
import pandas as pd

df = pd.read_csv("fraudTrain.csv")
df2 = pd.read_csv("fraudTest.csv")

In [11]:
df.head(10)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0
5,5,2019-01-01 00:04:08,4767265376804500,"fraud_Stroman, Hudson and Erdman",gas_transport,94.63,Jennifer,Conner,F,4655 David Island,...,40.3750,-75.2045,2158,Transport planner,1961-06-19,189a841a0a8ba03058526bcfe566aab5,1325376248,40.653382,-76.152667,0
6,6,2019-01-01 00:04:42,30074693890476,fraud_Rowe-Vandervort,grocery_net,44.54,Kelsey,Richards,F,889 Sarah Station Suite 624,...,37.9931,-100.9893,2691,Arboriculturist,1993-08-16,83ec1cc84142af6e2acf10c44949e720,1325376282,37.162705,-100.153370,0
7,7,2019-01-01 00:05:08,6011360759745864,fraud_Corwin-Collins,gas_transport,71.65,Steven,Williams,M,231 Flores Pass Suite 720,...,38.8432,-78.6003,6018,"Designer, multimedia",1947-08-21,6d294ed2cc447d2c71c7171a3d54967c,1325376308,38.948089,-78.540296,0
8,8,2019-01-01 00:05:18,4922710831011201,fraud_Herzog Ltd,misc_pos,4.27,Heather,Chase,F,6888 Hicks Stream Suite 954,...,40.3359,-79.6607,1472,Public affairs consultant,1941-03-07,fc28024ce480f8ef21a32d64c93a29f5,1325376318,40.351813,-79.958146,0
9,9,2019-01-01 00:06:01,2720830304681674,"fraud_Schoen, Kuphal and Nitzsche",grocery_pos,198.39,Melissa,Aguilar,F,21326 Taylor Squares Suite 708,...,36.5220,-87.3490,151785,Pathologist,1974-03-28,3b9014ea8fb80bd65de0b1463b00b00e,1325376361,37.179198,-87.485381,0


In [12]:
df2.head(10)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0
5,5,2020-06-21 12:15:37,30407675418785,fraud_Daugherty LLC,kids_pets,19.55,Danielle,Evans,F,76752 David Lodge Apt. 064,...,42.1939,-76.7361,520,Psychotherapist,1991-10-13,798db04aaceb4febd084f1a7c404da93,1371816937,41.747157,-77.584197,0
6,6,2020-06-21 12:15:44,213180742685905,fraud_Romaguera Ltd,health_fitness,133.93,Kayla,Sutton,F,010 Weaver Land,...,40.5070,-123.9743,1139,"Therapist, occupational",1951-01-15,17003d7ce534440eadb10c4750e020e5,1371816944,41.499458,-124.888729,0
7,7,2020-06-21 12:15:50,3589289942931264,fraud_Reichel LLC,personal_care,10.37,Paula,Estrada,F,350 Stacy Glens,...,43.7557,-97.5936,343,"Development worker, international aid",1972-03-05,8be473af4f05fc6146ea55ace73e7ca2,1371816950,44.495498,-97.728453,0
8,8,2020-06-21 12:16:10,3596357274378601,"fraud_Goyette, Howell and Collier",shopping_pos,4.37,David,Everett,M,4138 David Fall,...,41.0001,-78.2357,3688,Advice worker,1973-05-27,71a1da150d1ce510193d7622e08e784e,1371816970,41.546067,-78.120238,0
9,9,2020-06-21 12:16:11,3546897637165774,fraud_Kilback Group,food_dining,66.54,Kayla,Obrien,F,7921 Robert Port Suite 343,...,31.6591,-96.8094,263,Barrister,1956-05-30,a7915132c7c4240996ba03a47f81e3bd,1371816971,31.782919,-96.366185,0


In [13]:
TotalTransaction = df["amt"]
print(TotalTransaction[np.abs(TotalTransaction) > 1000])
print(df[np.abs(TotalTransaction) > 2000])


232        1055.47
511        1636.87
723        1047.52
824        1433.54
1480       1025.38
            ...   
1294390    1006.48
1294714    2090.14
1295108    1064.44
1295255    1063.03
1295491    1210.91
Name: amt, Length: 3936, dtype: float64
         Unnamed: 0 trans_date_trans_time               cc_num  \
1784           1784   2019-01-01 18:51:15      341546199006537   
5627           5627   2019-01-04 16:04:51     3589255887819806   
11469         11469   2019-01-07 18:46:44     3589255887819806   
12084         12084   2019-01-07 23:50:36     3567697931646329   
14254         14254   2019-01-08 22:41:26        4561546772499   
...             ...                   ...                  ...   
1292248     1292248   2020-06-19 18:15:43     3577794103155425   
1294000     1294000   2020-06-20 13:12:28     4003989662068504   
1294004     1294004   2020-06-20 13:14:23         630441765090   
1294387     1294387   2020-06-20 16:30:43  4536996888716062123   
1294714     1294714   202

In [14]:
TotalTransaction2 = df2["amt"]
print(TotalTransaction2[np.abs(TotalTransaction2) > 1000])
print(df2[np.abs(TotalTransaction2) > 2000])

167       1199.45
428       1881.53
951       3204.98
999       2068.05
1398      1420.90
           ...   
554526    3304.44
554553    1362.77
555004    2149.66
555229    1309.21
555637    1164.37
Name: amt, Length: 1583, dtype: float64
        Unnamed: 0 trans_date_trans_time               cc_num  \
951            951   2020-06-21 17:40:54  4292743669224718067   
999            999   2020-06-21 17:55:33     4933461930348832   
1428          1428   2020-06-21 20:31:19     5540636818935089   
6825          6825   2020-06-23 05:52:26     4239552724014407   
8851          8851   2020-06-23 18:10:43  4312133045694601139   
...            ...                   ...                  ...   
552225      552225   2020-12-30 23:21:11     4067137330196900   
552362      552362   2020-12-31 00:08:32      372246459334925   
553937      553937   2020-12-31 13:42:39       30596478689301   
554526      554526   2020-12-31 17:06:52     6528911529051375   
555004      555004   2020-12-31 19:43:11     43

## 5.3 Feature Scaling and Normalization

In [15]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

In [17]:
data = { 'Age': [25, 30, 45, 35, 22],'Salary': [50000, 64000, 120000, 85000, 1200000] } 
df = pd.DataFrame(data) 
print("Original Data:\n", df, "\n")

Original Data:
    Age   Salary
0   25    50000
1   30    64000
2   45   120000
3   35    85000
4   22  1200000 



In [18]:
min_max_scaler = MinMaxScaler()
df_min_max = pd.DataFrame(min_max_scaler.fit_transform(df),columns=df.columns)
print("Min-Max Scaled Data:\n", df_min_max, "\n")

Min-Max Scaled Data:
         Age    Salary
0  0.130435  0.000000
1  0.347826  0.012174
2  1.000000  0.060870
3  0.565217  0.030435
4  0.000000  1.000000 



In [19]:
standard_scaler = StandardScaler()
df_standard = pd.DataFrame(standard_scaler.fit_transform(df),columns=df.columns)
print("Standardized Data:\n", df_standard, "\n") 

Standardized Data:
         Age    Salary
0 -0.788742 -0.565609
1 -0.172537 -0.534409
2  1.676077 -0.409609
3  0.443667 -0.487609
4 -1.158465  1.997236 

